# Consolidated Empirical Report

**F-Score on High B/M Equities with MPT Portfolio Construction across Three Equity Markets**
WorldQuant University — MScFE 690 Capstone Project

---

Every table and figure the paper will contain, composed from `../results/` and written to
`resources/` ready for the LaTeX build. Row and column labels are the ones the paper uses;
numbers are formatted once; a statistic that beats its matched random null at the 5% level is
bold with an asterisk and carries its own p-value in the column beside it.

Each table names the `[PLACEHOLDER: ...]` in the draft it fills, so the transfer is a lookup.

Re-run this notebook after anything upstream changes and `resources/` is rebuilt from scratch.

**The review companion lives in `consolidated_report_notes.ipynb`** — open items, figure audit,
build manifest, and the placeholders that still need prose. Nothing in this notebook is a
to-do list; it is the output.

### How to run

```
jupyter nbconvert --to notebook --execute --inplace consolidated_report.ipynb
```

### Output layout

```
resources/
  _all_tables.tex   \input{} lines for every table below, in section order
  tables/           <name>.tex (booktabs) + <name>.csv
  figures/          paper-ready PNGs
  notes/            written by the review companion, never part of the paper
```

In [1]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path.cwd()))

import pandas as pd

import report_lib as R
from report_lib import (ALPHA, DASH, DIAG_LABEL, MARKETS, MARKET_LABEL,
                        STAR_LEVELS, STRATEGY_LABEL, copy_figure, emit, integer,
                        marked, num, pct, pm, pval, read, read_indexed,
                        read_placement, sig, stars)

# Start from an empty resources/ so a re-run can never leave a stale table
# behind from a specification that has since been dropped.
for _stale in list(R.TABLES.glob("*")) + list(R.FIGURES.glob("*")) + [R.RES / "_all_tables.tex"]:
    if _stale.is_file():
        _stale.unlink()

pd.set_option("display.width", 200)
print(f"results   : {R.RESULTS}")
print(f"resources : {R.RES}")
print(f"alpha     : {ALPHA:.2f}  (one-sided empirical p-value, verdict is p < alpha)")
print(f"stars     : {', '.join(f'{m} for p < {lv:g}' for lv, m in STAR_LEVELS)}")

results   : /Users/lap14821-local/Documents/Software Development/worldquant_uni/fscore_capstone/results
resources : /Users/lap14821-local/Documents/Software Development/worldquant_uni/fscore_capstone/consolidated_report/resources
alpha     : 0.05  (one-sided empirical p-value, verdict is p < alpha)
stars     : * for p < 0.05


In [2]:
# ---------------------------------------------------------------------
# Table A — conventions frozen by the unified code
# Fills: P08 (K / tie-break seed), P11 (turnover formula), P12 (Monte Carlo N),
#        P13 (drawdown sign), P14 (risk-free rate), and part of P04/P05.
# Every row was read off the implementation, not the draft.
# ---------------------------------------------------------------------
conventions = pd.DataFrame(
    [
        ("Significance level",
         "5%, one-sided empirical p-value; verdict is $p < 0.05$ (strict, so $p = 0.050$ is not significant)",
         "evaluation/backtest.py:15,132"),
        ("Primary measure",
         "Gross Sharpe ratio, fixed in advance",
         "grid.py:29"),
        ("Risk-free rate",
         "$r_f = 0$ in every Sharpe ratio, all three markets",
         "evaluation/backtest.py:93"),
        ("Maximum drawdown sign",
         "Negative return (trough / peak $-$ 1); the Monte Carlo test treats higher as better",
         "evaluation/backtest.py:101"),
        ("Turnover",
         r"One-way, $\frac{1}{2}\sum_i |w^{new}_i - w^{old}_i|$, on drifted pre-rebalance weights",
         "evaluation/backtest.py:108"),
        ("Formation and holding",
         "Formed 1 July, held one complete year to 30 June, no intra-year trades",
         "pipeline.py:148"),
        ("Covariance lookback",
         "36 months of daily returns ending the day before formation",
         "pipeline.py:46"),
        ("Covariance cleaning",
         "RMT denoising (Marchenko--Pastur band flattened); detoning OFF for any inverse",
         "construction/weights.py:23"),
        ("Tie handling",
         "Random tie-break on the integer F-Score, seeded per formation year ($seed + year$)",
         "selection/baskets.rank_by_fscore"),
        ("High-B/M cutoff",
         "Top 40% by book-to-market, same fraction in every market",
         "pipeline.py:141 (value_quantile=0.4)"),
        ("Eligible universe",
         "Top 150 names by median daily dollar volume, after the listing and fundamentals screens",
         "pipeline.py:105,141"),
        ("Monte Carlo draws",
         sig("1000 for equal-weight, 300 for GMV — not one common $N$", True),
         "*_mc_placement.csv (n_draws)"),
    ],
    columns=["Convention in the unified code", "Source"] and
            ["Item", "Convention in the unified code", "Source"],
).set_index("Item")

emit(conventions,
     name="table_A_frozen_conventions",
     number="Table A",
     title="Conventions frozen by the unified pipeline",
     placeholder="P08, P11, P12, P13, P14",
     section="§3.8, §6",
     index_header="Item",
     notes=("Read from the implementation rather than the draft. The starred row is the one "
            "convention the draft requires but the code does not yet satisfy: §6 asks for a "
            "single Monte Carlo $N$ across all headline tests."))

**Table A. Conventions frozen by the unified pipeline**  
<sub>§3.8, §6 &middot; fills P08, P11, P12, P13, P14 &middot; `resources/tables/table_A_frozen_conventions.tex`</sub>

Item,Convention in the unified code,Source
Significance level,"5%, one-sided empirical p-value; verdict is $p < 0.05$ (strict, so $p = 0.050$ is not significant)","evaluation/backtest.py:15,132"
Primary measure,"Gross Sharpe ratio, fixed in advance",grid.py:29
Risk-free rate,"$r_f = 0$ in every Sharpe ratio, all three markets",evaluation/backtest.py:93
Maximum drawdown sign,Negative return (trough / peak $-$ 1); the Monte Carlo test treats higher as better,evaluation/backtest.py:101
Turnover,"One-way, $\frac{1}{2}\sum_i |w^{new}_i - w^{old}_i|$, on drifted pre-rebalance weights",evaluation/backtest.py:108
Formation and holding,"Formed 1 July, held one complete year to 30 June, no intra-year trades",pipeline.py:148
Covariance lookback,36 months of daily returns ending the day before formation,pipeline.py:46
Covariance cleaning,RMT denoising (Marchenko--Pastur band flattened); detoning OFF for any inverse,construction/weights.py:23
Tie handling,"Random tie-break on the integer F-Score, seeded per formation year ($seed + year$)",selection/baskets.rank_by_fscore
High-B/M cutoff,"Top 40% by book-to-market, same fraction in every market",pipeline.py:141 (value_quantile=0.4)


<sub>*Notes.* Read from the implementation rather than the draft. The starred row is the one convention the draft requires but the code does not yet satisfy: §6 asks for a single Monte Carlo $N$ across all headline tests.</sub>

,Convention in the unified code,Source
Item,,
Significance level,"5%, one-sided empirical p-value; verdict is $p...","evaluation/backtest.py:15,132"
Primary measure,"Gross Sharpe ratio, fixed in advance",grid.py:29
Risk-free rate,"$r_f = 0$ in every Sharpe ratio, all three mar...",evaluation/backtest.py:93
Maximum drawdown sign,Negative return (trough / peak $-$ 1); the Mon...,evaluation/backtest.py:101
Turnover,"One-way, $\frac{1}{2}\sum_i |w^{new}_i - w^{ol...",evaluation/backtest.py:108
Formation and holding,"Formed 1 July, held one complete year to 30 Ju...",pipeline.py:148
Covariance lookback,36 months of daily returns ending the day befo...,pipeline.py:46
Covariance cleaning,RMT denoising (Marchenko--Pastur band flattene...,construction/weights.py:23
Tie handling,"Random tie-break on the integer F-Score, seede...",selection/baskets.rank_by_fscore


---
## 3.8 Unified design decisions

Table 3.1 of the draft, restated with a third column that records what the code does rather than
what the team intended. Rows where the two differ are the ones to settle before the LaTeX pass.

In [3]:
design = pd.DataFrame(
    [
        ("Markets", "United States, Japan, Vietnam", "Final"),
        ("Primary opportunity set", "High-B/M subset formed before F-Score ranking", "Final"),
        ("High-B/M cutoff", "Top 40% by book-to-market, identical in every market", "Verified in code"),
        ("Financial firms", "Excluded upstream, in the input panels, not by the pipeline",
         sig("Not logged as a stage", True)),
        ("Missing F-Score inputs", "Firm-year removed; a missing signal is never scored as zero", "Verified in code"),
        ("Formation date", "1 July", "Verified in code"),
        ("Holding period", "One complete year, through 30 June", "Verified in code"),
        ("Rebalancing", "Once per year; no trades during the holding year", "Verified in code"),
        ("Turnover base", "Drifted pre-rebalance weights, union of old and new holdings", "Verified in code"),
        ("Headline returns", "Gross; turnover and cost drag reported separately", "Verified in code"),
        ("Primary basket size", "$K = 30$ in the main study; $K = 25$ in the grid",
         sig("Two values in use", True)),
        ("Robustness basket sizes", "$K \\in \\{20, 25, 30\\}$", "Run (grid)"),
        ("Covariance cleaning", "RMT denoising; detoning off for any inverse", "Verified in code"),
        ("Covariance lookback", "36 months of daily returns", "Verified in code"),
        ("GMV constraints", "Long-only, fully invested; sector-capped variant also produced",
         sig("Headline set undecided", True)),
        ("Tie handling", "Random tie-break under one seed per formation year", "Verified in code"),
        ("Monte Carlo universe", "The same annual high-B/M universe as the F-Score basket", "Verified in code"),
        ("Primary Monte Carlo $N$", "1000 for equal-weight, 300 for GMV",
         sig("Not one common $N$", True)),
        ("Robustness $N$ grid", "$N \\in \\{1000, 2000, 5000\\}$", "Run (grid)"),
        ("Significance level", "5%, one-sided, strict $p < 0.05$", "Verified in code"),
        ("Delisting", "Exit at the last tradable price with volume", "Verified in code"),
        ("$-100\\%$ delisting case", "Robustness only", sig("Not run", True)),
        ("Risk-free rate", "$r_f = 0$ in every Sharpe ratio", "Verified in code"),
        ("Cross-country period", "One common period for the headline comparison",
         sig("No common period exists yet", True)),
        ("Full-country period", "Reported separately as within-country robustness", "Run"),
    ],
    columns=["Design item", "Main convention", "Status in the unified code"],
).set_index("Design item")

emit(design,
     name="table_3_1_design_decisions",
     number="Table 3.1",
     title="Unified research-design decisions for the cross-country analysis",
     placeholder="P04, P05, P08, P10, P12",
     section="§3.8",
     index_header="Design item",
     notes=("Starred entries are the ones where the implementation does not yet match the "
            "stated convention; they are itemised in the review companion."))

**Table 3.1. Unified research-design decisions for the cross-country analysis**  
<sub>§3.8 &middot; fills P04, P05, P08, P10, P12 &middot; `resources/tables/table_3_1_design_decisions.tex`</sub>

Design item,Main convention,Status in the unified code
Markets,"United States, Japan, Vietnam",Final
Primary opportunity set,High-B/M subset formed before F-Score ranking,Final
High-B/M cutoff,"Top 40% by book-to-market, identical in every market",Verified in code
Financial firms,"Excluded upstream, in the input panels, not by the pipeline",Not logged as a stage*
Missing F-Score inputs,Firm-year removed; a missing signal is never scored as zero,Verified in code
Formation date,1 July,Verified in code
Holding period,"One complete year, through 30 June",Verified in code
Rebalancing,Once per year; no trades during the holding year,Verified in code
Turnover base,"Drifted pre-rebalance weights, union of old and new holdings",Verified in code
Headline returns,Gross; turnover and cost drag reported separately,Verified in code


<sub>*Notes.* Starred entries are the ones where the implementation does not yet match the stated convention; they are itemised in the review companion.</sub>

,Main convention,Status in the unified code
Design item,,
Markets,"United States, Japan, Vietnam",Final
Primary opportunity set,High-B/M subset formed before F-Score ranking,Final
High-B/M cutoff,"Top 40% by book-to-market, identical in every ...",Verified in code
Financial firms,"Excluded upstream, in the input panels, not by...",Not logged as a stage*
Missing F-Score inputs,Firm-year removed; a missing signal is never s...,Verified in code
Formation date,1 July,Verified in code
Holding period,"One complete year, through 30 June",Verified in code
Rebalancing,Once per year; no trades during the holding year,Verified in code
Turnover base,"Drifted pre-rebalance weights, union of old an...",Verified in code


---
## 7. Empirical Results

Before the country tables, one reconciliation. The draft speaks of a *common period* shared by
the three markets and a *full period* reported per country. The two windows currently produced
are not those: each market's headline window is its own, and Japan's is two formation years.
Table 7.0 states the windows that actually exist, so no country table below is read as covering
a period it does not.

In [4]:
# ---------------------------------------------------------------------
# Table 7.0 — the windows that actually exist
# Built from robustness_full_period.csv (one row per market x strategy) and
# the per-year diagnostics.
# ---------------------------------------------------------------------
rob = read(R.RESULTS / "robustness_full_period.csv")

rows = {}
for m in MARKETS:
    diag = read(R.RESULTS / f"{m}_diagnostics.csv")
    full = read(R.RESULTS / f"{m}_fullperiod_diagnostics.csv")
    r = rob[rob.market == m].iloc[0] if rob is not None and (rob.market == m).any() else None
    rows[MARKET_LABEL[m]] = {
        "Headline first formation": integer(diag.year.min()) if diag is not None else DASH,
        "Headline last formation": integer(diag.year.max()) if diag is not None else DASH,
        "Headline formations": sig(integer(len(diag)), len(diag) < 5) if diag is not None else DASH,
        "Full-window first formation": integer(full.year.min()) if full is not None else DASH,
        "Full-window last formation": integer(full.year.max()) if full is not None else DASH,
        "Full-window formations": integer(len(full)) if full is not None else DASH,
        "Full-window span": (r.full_span if r is not None else DASH),
    }
windows = pd.DataFrame(rows).T
windows.index.name = "Market"

emit(windows,
     name="table_7_0_sample_windows",
     number="Table 7.0",
     title="Formation-year windows actually produced, by market",
     placeholder="P03",
     section="§3.6, §7",
     index_header="Market",
     notes=("The intersection of the three headline windows is 2023--2024, two formation years. "
            "Japan is the binding constraint: its fundamentals cache carries no point-in-time "
            "statements before fiscal 2022 (see Table 11.2). Any statement in §8 that compares "
            "the three markets over a common period is limited to those two years."))

**Table 7.0. Formation-year windows actually produced, by market**  
<sub>§3.6, §7 &middot; fills P03 &middot; `resources/tables/table_7_0_sample_windows.tex`</sub>

Market,Headline first formation,Headline last formation,Headline formations,Full-window first formation,Full-window last formation,Full-window formations,Full-window span
United States,2012,2024,13,2010,2024,15,2010-07 to 2025-06
Japan,2023,2024,2*,2023,2024,2,2023-07 to 2025-06
Vietnam,2012,2024,13,2011,2024,14,2011-07 to 2025-06


<sub>*Notes.* The intersection of the three headline windows is 2023--2024, two formation years. Japan is the binding constraint: its fundamentals cache carries no point-in-time statements before fiscal 2022 (see Table 11.2). Any statement in §8 that compares the three markets over a common period is limited to those two years.</sub>

,Headline first formation,Headline last formation,Headline formations,Full-window first formation,Full-window last formation,Full-window formations,Full-window span
Market,,,,,,,
United States,2012,2024,13,2010,2024,15,2010-07 to 2025-06
Japan,2023,2024,2*,2023,2024,2,2023-07 to 2025-06
Vietnam,2012,2024,13,2011,2024,14,2011-07 to 2025-06


In [5]:
# ---------------------------------------------------------------------
# Shared composers for the per-country tables.
# Kept here rather than in report_lib.py so the team can adjust the row
# order, the labels, or the column set without leaving the notebook.
#
# Layout follows the convention of the published F-Score papers (Ng and Shen
# 2016, Table 6): every tested statistic is followed immediately by its own
# p-value against the matched random null, and carries its own stars — no
# number inherits the verdict of another. Descriptive quantities that have no
# null attached sit together at the right, unstarred.
# ---------------------------------------------------------------------
FUNNEL_COLS = ["universe", "value_set", "scored", "k",
               "delisted_in_holding_year", "tie_break_slots",
               "fscore_mean", "fscore_basket_min"]


def funnel_table(market: str) -> pd.DataFrame:
    """Formation-year counts at every stage that the pipeline logs."""
    diag = read(R.RESULTS / f"{market}_diagnostics.csv")
    if diag is None:
        return pd.DataFrame()
    out = pd.DataFrame(index=[integer(y) for y in diag.year])
    out.index.name = "Formation year"
    for col in FUNNEL_COLS:
        if col not in diag:
            continue
        label = DIAG_LABEL[col]
        if col == "fscore_mean":
            out[label] = [num(v, 2) for v in diag[col]]
        elif col == "k":
            # flag a year that could not fill the nominal basket
            kmax = diag[col].max()
            out[label] = [sig(integer(v), v < kmax) for v in diag[col]]
        else:
            out[label] = [integer(v) for v in diag[col]]
    return out


PERF_ORDER = ["fscore_EW", "fscore_GMV", "fscore_GMVsec"]

# metrics that carry a Monte Carlo null, in the order the paper reads them
TESTED = [("ann_return", "Annualised return", pct),
          ("sharpe", "Sharpe ratio", num),
          ("max_drawdown", "Maximum drawdown", pct)]
# quantities with no null attached
DESCRIPTIVE = [("ann_vol", "Annualised volatility", pct),
               ("turnover", "Turnover (one-way)", pct),
               ("effective_n", "Effective N", lambda v: num(v, 1))]

PERF_COLS = ([c for _, label, _ in TESTED for c in (label, "p-value")]
             + [label for _, label, _ in DESCRIPTIVE])


def _placement(plc, strategy, metric, field):
    if plc is None or (strategy, metric) not in plc.index:
        return float("nan")
    return plc.loc[(strategy, metric), field]


def performance_table(market: str, window: str = "headline") -> pd.DataFrame:
    """One market's 2x2 performance panel: F-Score vs matched random,
    equal-weight vs GMV, with the market proxies underneath. Each tested
    statistic is followed by its Monte Carlo z-statistic."""
    stem = market if window == "headline" else f"{market}_fullperiod"
    summ = read_indexed(R.RESULTS / f"{stem}_summary.csv")
    plc = read_placement(R.RESULTS / f"{stem}_placement.csv"
                         if window != "headline"
                         else R.RESULTS / f"{market}_mc_placement.csv")
    if summ is None:
        return pd.DataFrame()

    labels, rows = [], []
    for strat in PERF_ORDER:
        if strat not in summ.index:
            continue
        s = summ.loc[strat]

        row = []
        for key, _, fmt in TESTED:
            p = _placement(plc, strat, key, "p_value")
            # the statistic carries the star; the column beside it gives the
            # empirical p-value against the matched random null
            row += [marked(fmt(s[key]), p), num(p, 3)]
        row += [fmt(s[key]) for key, _, fmt in DESCRIPTIVE]
        labels.append(STRATEGY_LABEL[strat])
        rows.append(row)

        # the matched random null for the same construction: it is the
        # reference point, so it has no distance from itself to report
        null = []
        for key, _, fmt in TESTED:
            null.append(pm(_placement(plc, strat, key, "random_mean"),
                           _placement(plc, strat, key, "random_std"), fmt))
            null.append(DASH)
        null += [DASH] * len(DESCRIPTIVE)
        labels.append(STRATEGY_LABEL["random_" + strat.split("_", 1)[1]])
        rows.append(null)

    for ix in summ.index:                       # market proxies
        if ix in STRATEGY_LABEL:
            continue
        s = summ.loc[ix]
        proxy = []
        for key, _, fmt in TESTED:
            proxy += [fmt(s[key]), DASH]
        proxy += [pct(s.ann_vol), DASH, DASH]
        labels.append(ix)
        rows.append(proxy)

    df = pd.DataFrame(rows, index=labels, columns=PERF_COLS)
    df.index.name = "Portfolio"
    return df


def mc_draws_note(market: str) -> str:
    plc = read_placement(R.RESULTS / f"{market}_mc_placement.csv")
    if plc is None:
        return ""
    by = plc.reset_index().groupby("strategy").n_draws.max()
    parts = [f"{STRATEGY_LABEL.get(k, k)}: $N = {int(v)}$" for k, v in by.items()]
    return "Monte Carlo draws — " + "; ".join(parts) + ". "


STAR_NOTE = (
    "An asterisk marks significance at the 5% level, the single level fixed in "
    "advance in §6; each statistic is starred on its own empirical p-value, not "
    "on the Sharpe ratio's. "
)

P_NOTE = (
    "Each tested statistic is followed by its one-sided empirical p-value against the "
    "matched random null: the share of Monte Carlo draws that equalled or beat the "
    "F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its "
    "test treats a shallower drawdown as better. "
)

PERF_NOTES = (
    "Rows pair each F-Score portfolio with the matched random portfolio drawn from the same "
    "annual high-B/M universe under the same construction, so reading across a pair isolates "
    "the selection rule and reading down the pairs isolates portfolio construction. Random rows "
    "report the Monte Carlo mean $\\pm$ standard deviation and carry no test of their own, "
    "being the null. " + P_NOTE + STAR_NOTE +
    "Annualised volatility, turnover and Effective N have no null attached and are therefore "
    "never starred. Returns are gross; turnover is one-way."
)
print("composers ready — every tested statistic carries its own p-value and stars")


composers ready — every tested statistic carries its own p-value and stars


---
### 7.1 United States

In [6]:
funnel_us = funnel_table("us")
emit(funnel_us,
     name="table_7_1a_us_funnel",
     number="Table 7.1a",
     title="United States: formation-year counts at every logged stage",
     placeholder="P15",
     section="§7.1",
     index_header="Formation year",
     notes=("Two stages the draft asks for are not logged by the pipeline and cannot be shown: "
            "the raw universe before the liquidity cap (the eligible-universe column is already "
            "the top 150 names by median daily dollar volume, so it is capped rather than "
            "counted), and the number of selected names actually priced through the holding "
            "year. Delisted-in-holding-year is the closest available proxy for the latter. "
            "A starred K marks a year that could not fill the nominal basket."))

**Table 7.1a. United States: formation-year counts at every logged stage**  
<sub>§7.1 &middot; fills P15 &middot; `resources/tables/table_7_1a_us_funnel.tex`</sub>

Formation year,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
2012,150,60,60,30,0,17,5.73,6
2013,150,60,60,30,0,18,5.43,6
2014,150,60,60,30,0,11,5.67,6
2015,150,60,59,30,0,13,5.69,6
2016,150,60,60,30,0,10,4.97,5
2017,150,60,59,30,0,18,5.37,6
2018,150,60,60,30,0,16,5.62,6
2019,150,60,52,30,0,13,6.13,6
2020,150,60,58,30,0,15,5.52,6
2021,150,60,59,30,0,5,5.17,5


<sub>*Notes.* Two stages the draft asks for are not logged by the pipeline and cannot be shown: the raw universe before the liquidity cap (the eligible-universe column is already the top 150 names by median daily dollar volume, so it is capped rather than counted), and the number of selected names actually priced through the holding year. Delisted-in-holding-year is the closest available proxy for the latter. A starred K marks a year that could not fill the nominal basket.</sub>

,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
Formation year,,,,,,,,
2012,150,60,60,30,0,17,5.73,6
2013,150,60,60,30,0,18,5.43,6
2014,150,60,60,30,0,11,5.67,6
2015,150,60,59,30,0,13,5.69,6
2016,150,60,60,30,0,10,4.97,5
2017,150,60,59,30,0,18,5.37,6
2018,150,60,60,30,0,16,5.62,6
2019,150,60,52,30,0,13,6.13,6
2020,150,60,58,30,0,15,5.52,6


In [7]:
perf_us = performance_table("us", "headline")
emit(perf_us,
     name="table_7_1b_us_performance_headline",
     number="Table 7.1b",
     title="United States: performance over the headline window",
     placeholder="P16",
     section="§7.1",
     index_header="Portfolio",
     notes=mc_draws_note("us") + PERF_NOTES)

**Table 7.1b. United States: performance over the headline window**  
<sub>§7.1 &middot; fills P16 &middot; `resources/tables/table_7_1b_us_performance_headline.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",15.92%,0.565,0.811,0.527,-38.64%,0.102,19.62%,60.94%,30.0
"Matched random, equal-weight",16.16% ± 1.28%,—,0.817 ± 0.065,—,-41.38% ± 2.17%,—,—,—,—
"F-Score, GMV (RMT-denoised)",11.41%,0.087,0.736*,0.030,-29.11%*,0.040,15.49%,60.96%,5.6
"Matched random, GMV",9.08% ± 1.72%,—,0.550 ± 0.107,—,-37.55% ± 3.70%,—,—,—,—
"F-Score, sector-capped GMV",15.14%*,0.003,0.918*,0.003,-38.56%,0.477,16.49%,67.06%,8.8
"Matched random, sector-capped GMV",10.87% ± 1.71%,—,0.644 ± 0.103,—,-38.97% ± 3.36%,—,—,—,—
SPY (S&P 500),14.38%,—,0.847,—,-33.72%,—,16.97%,—,—
VTV (US value ETF),12.10%,—,0.756,—,-36.78%,—,15.99%,—,—


<sub>*Notes.* Monte Carlo draws — F-Score, equal-weight: $N = 1000$; F-Score, GMV (RMT-denoised): $N = 300$; F-Score, sector-capped GMV: $N = 300$. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",15.92%,0.565,0.811,0.527,-38.64%,0.102,19.62%,60.94%,30.0
"Matched random, equal-weight",16.16% $\pm$ 1.28%,—,0.817 $\pm$ 0.065,—,-41.38% $\pm$ 2.17%,—,—,—,—
"F-Score, GMV (RMT-denoised)",11.41%,0.087,0.736*,0.030,-29.11%*,0.040,15.49%,60.96%,5.6
"Matched random, GMV",9.08% $\pm$ 1.72%,—,0.550 $\pm$ 0.107,—,-37.55% $\pm$ 3.70%,—,—,—,—
"F-Score, sector-capped GMV",15.14%*,0.003,0.918*,0.003,-38.56%,0.477,16.49%,67.06%,8.8
"Matched random, sector-capped GMV",10.87% $\pm$ 1.71%,—,0.644 $\pm$ 0.103,—,-38.97% $\pm$ 3.36%,—,—,—,—
SPY (S&P 500),14.38%,—,0.847,—,-33.72%,—,16.97%,—,—
VTV (US value ETF),12.10%,—,0.756,—,-36.78%,—,15.99%,—,—


In [8]:
perf_us_full = performance_table("us", "full")
emit(perf_us_full,
     name="table_7_1c_us_performance_full",
     number="Table 7.1c",
     title="United States: performance over the full available window",
     placeholder="P16",
     section="§7.1",
     index_header="Portfolio",
     notes=("Same construction as the previous table over every formation year the data "
            "support; see Table 7.0 for the two windows. " + PERF_NOTES))

**Table 7.1c. United States: performance over the full available window**  
<sub>§7.1 &middot; fills P16 &middot; `resources/tables/table_7_1c_us_performance_full.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",16.68%,0.605,0.849,0.569,-38.64%,0.102,19.65%,62.78%,30.0
"Matched random, equal-weight",17.02% ± 1.16%,—,0.858 ± 0.060,—,-41.38% ± 2.17%,—,—,—,—
"F-Score, GMV (RMT-denoised)",12.46%,0.113,0.815*,0.040,-29.11%*,0.040,15.28%,64.79%,5.4
"Matched random, GMV",10.42% ± 1.56%,—,0.642 ± 0.100,—,-37.55% ± 3.70%,—,—,—,—
"F-Score, sector-capped GMV",15.71%*,0.027,0.957*,0.020,-38.56%,0.477,16.41%,68.41%,8.6
"Matched random, sector-capped GMV",12.43% ± 1.60%,—,0.746 ± 0.098,—,-38.97% ± 3.36%,—,—,—,—


<sub>*Notes.* Same construction as the previous table over every formation year the data support; see Table 7.0 for the two windows. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",16.68%,0.605,0.849,0.569,-38.64%,0.102,19.65%,62.78%,30.0
"Matched random, equal-weight",17.02% $\pm$ 1.16%,—,0.858 $\pm$ 0.060,—,-41.38% $\pm$ 2.17%,—,—,—,—
"F-Score, GMV (RMT-denoised)",12.46%,0.113,0.815*,0.040,-29.11%*,0.040,15.28%,64.79%,5.4
"Matched random, GMV",10.42% $\pm$ 1.56%,—,0.642 $\pm$ 0.100,—,-37.55% $\pm$ 3.70%,—,—,—,—
"F-Score, sector-capped GMV",15.71%*,0.027,0.957*,0.020,-38.56%,0.477,16.41%,68.41%,8.6
"Matched random, sector-capped GMV",12.43% $\pm$ 1.60%,—,0.746 $\pm$ 0.098,—,-38.97% $\pm$ 3.36%,—,—,—,—


---
### 7.2 Japan

In [9]:
funnel_japan = funnel_table("japan")
emit(funnel_japan,
     name="table_7_2a_japan_funnel",
     number="Table 7.2a",
     title="Japan: formation-year counts at every logged stage",
     placeholder="P18",
     section="§7.2",
     index_header="Formation year",
     notes=("Two stages the draft asks for are not logged by the pipeline and cannot be shown: "
            "the raw universe before the liquidity cap (the eligible-universe column is already "
            "the top 150 names by median daily dollar volume, so it is capped rather than "
            "counted), and the number of selected names actually priced through the holding "
            "year. Delisted-in-holding-year is the closest available proxy for the latter. "
            "A starred K marks a year that could not fill the nominal basket."))

**Table 7.2a. Japan: formation-year counts at every logged stage**  
<sub>§7.2 &middot; fills P18 &middot; `resources/tables/table_7_2a_japan_funnel.tex`</sub>

Formation year,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
2023,118,47,9,9*,0,1,6.44,3
2024,150,60,60,30,0,2,6.17,6


<sub>*Notes.* Two stages the draft asks for are not logged by the pipeline and cannot be shown: the raw universe before the liquidity cap (the eligible-universe column is already the top 150 names by median daily dollar volume, so it is capped rather than counted), and the number of selected names actually priced through the holding year. Delisted-in-holding-year is the closest available proxy for the latter. A starred K marks a year that could not fill the nominal basket.</sub>

,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
Formation year,,,,,,,,
2023,118,47,9,9*,0,1,6.44,3
2024,150,60,60,30,0,2,6.17,6


In [10]:
perf_japan = performance_table("japan", "headline")
emit(perf_japan,
     name="table_7_2b_japan_performance_headline",
     number="Table 7.2b",
     title="Japan: performance over the headline window",
     placeholder="P19",
     section="§7.2",
     index_header="Portfolio",
     notes=mc_draws_note("japan") + PERF_NOTES)

**Table 7.2b. Japan: performance over the headline window**  
<sub>§7.2 &middot; fills P19 &middot; `resources/tables/table_7_2b_japan_performance_headline.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,23.46%,96.67%,19.5
"Matched random, equal-weight",13.90% ± 5.17%,—,0.616 ± 0.224,—,-23.14% ± 1.64%,—,—,—,—
"F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,20.27%,100.00%,6.5
"Matched random, GMV",16.50% ± 6.96%,—,0.855 ± 0.351,—,-16.37% ± 2.89%,—,—,—,—
"F-Score, sector-capped GMV",23.51%,0.137,1.134,0.193,-16.39%,0.537,20.74%,100.00%,7.8
"Matched random, sector-capped GMV",16.70% ± 6.06%,—,0.867 ± 0.309,—,-16.58% ± 2.17%,—,—,—,—
"1306.T (TOPIX ETF, JPY)",13.96%,—,0.651,—,-22.82%,—,21.42%,—,—
"EWJV (MSCI Japan Value ETF, USD)",16.79%,—,0.912,—,-14.61%,—,18.41%,—,—


<sub>*Notes.* Monte Carlo draws — F-Score, equal-weight: $N = 1000$; F-Score, GMV (RMT-denoised): $N = 300$; F-Score, sector-capped GMV: $N = 300$. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,23.46%,96.67%,19.5
"Matched random, equal-weight",13.90% $\pm$ 5.17%,—,0.616 $\pm$ 0.224,—,-23.14% $\pm$ 1.64%,—,—,—,—
"F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,20.27%,100.00%,6.5
"Matched random, GMV",16.50% $\pm$ 6.96%,—,0.855 $\pm$ 0.351,—,-16.37% $\pm$ 2.89%,—,—,—,—
"F-Score, sector-capped GMV",23.51%,0.137,1.134,0.193,-16.39%,0.537,20.74%,100.00%,7.8
"Matched random, sector-capped GMV",16.70% $\pm$ 6.06%,—,0.867 $\pm$ 0.309,—,-16.58% $\pm$ 2.17%,—,—,—,—
"1306.T (TOPIX ETF, JPY)",13.96%,—,0.651,—,-22.82%,—,21.42%,—,—
"EWJV (MSCI Japan Value ETF, USD)",16.79%,—,0.912,—,-14.61%,—,18.41%,—,—


In [11]:
perf_japan_full = performance_table("japan", "full")
emit(perf_japan_full,
     name="table_7_2c_japan_performance_full",
     number="Table 7.2c",
     title="Japan: performance over the full available window",
     placeholder="P19",
     section="§7.2",
     index_header="Portfolio",
     notes=("Same construction as the previous table over every formation year the data "
            "support; see Table 7.0 for the two windows. " + PERF_NOTES))

**Table 7.2c. Japan: performance over the full available window**  
<sub>§7.2 &middot; fills P19 &middot; `resources/tables/table_7_2c_japan_performance_full.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,23.46%,96.67%,19.5
"Matched random, equal-weight",13.90% ± 5.17%,—,0.616 ± 0.224,—,-23.14% ± 1.64%,—,—,—,—
"F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,20.27%,100.00%,6.5
"Matched random, GMV",16.50% ± 6.96%,—,0.855 ± 0.351,—,-16.37% ± 2.89%,—,—,—,—
"F-Score, sector-capped GMV",23.51%,0.137,1.134,0.193,-16.39%,0.537,20.74%,100.00%,7.8
"Matched random, sector-capped GMV",16.70% ± 6.06%,—,0.867 ± 0.309,—,-16.58% ± 2.17%,—,—,—,—


<sub>*Notes.* Same construction as the previous table over every formation year the data support; see Table 7.0 for the two windows. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,23.46%,96.67%,19.5
"Matched random, equal-weight",13.90% $\pm$ 5.17%,—,0.616 $\pm$ 0.224,—,-23.14% $\pm$ 1.64%,—,—,—,—
"F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,20.27%,100.00%,6.5
"Matched random, GMV",16.50% $\pm$ 6.96%,—,0.855 $\pm$ 0.351,—,-16.37% $\pm$ 2.89%,—,—,—,—
"F-Score, sector-capped GMV",23.51%,0.137,1.134,0.193,-16.39%,0.537,20.74%,100.00%,7.8
"Matched random, sector-capped GMV",16.70% $\pm$ 6.06%,—,0.867 $\pm$ 0.309,—,-16.58% $\pm$ 2.17%,—,—,—,—


---
### 7.3 Vietnam

In [12]:
funnel_vietnam = funnel_table("vietnam")
emit(funnel_vietnam,
     name="table_7_3a_vietnam_funnel",
     number="Table 7.3a",
     title="Vietnam: formation-year counts at every logged stage",
     placeholder="P21",
     section="§7.3",
     index_header="Formation year",
     notes=("Two stages the draft asks for are not logged by the pipeline and cannot be shown: "
            "the raw universe before the liquidity cap (the eligible-universe column is already "
            "the top 150 names by median daily dollar volume, so it is capped rather than "
            "counted), and the number of selected names actually priced through the holding "
            "year. Delisted-in-holding-year is the closest available proxy for the latter. "
            "A starred K marks a year that could not fill the nominal basket."))

**Table 7.3a. Vietnam: formation-year counts at every logged stage**  
<sub>§7.3 &middot; fills P21 &middot; `resources/tables/table_7_3a_vietnam_funnel.tex`</sub>

Formation year,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
2012,150,60,52,30,2,6,4.33,4
2013,150,60,56,30,0,12,4.82,5
2014,150,60,57,30,2,12,4.81,5
2015,150,60,57,30,0,4,5.51,5
2016,150,60,51,30,0,6,5.20,5
2017,150,60,53,30,1,6,5.17,5
2018,150,60,57,30,1,7,5.30,5
2019,150,60,58,30,1,13,5.64,6
2020,150,60,58,30,0,5,5.40,5
2021,150,60,59,30,0,4,5.19,5


<sub>*Notes.* Two stages the draft asks for are not logged by the pipeline and cannot be shown: the raw universe before the liquidity cap (the eligible-universe column is already the top 150 names by median daily dollar volume, so it is capped rather than counted), and the number of selected names actually priced through the holding year. Delisted-in-holding-year is the closest available proxy for the latter. A starred K marks a year that could not fill the nominal basket.</sub>

,Eligible universe,High-B/M set,Scoreable,Selected K,Delisted in holding year,Tie-break slots,Mean F-Score,Min F-Score in basket
Formation year,,,,,,,,
2012,150,60,52,30,2,6,4.33,4
2013,150,60,56,30,0,12,4.82,5
2014,150,60,57,30,2,12,4.81,5
2015,150,60,57,30,0,4,5.51,5
2016,150,60,51,30,0,6,5.20,5
2017,150,60,53,30,1,6,5.17,5
2018,150,60,57,30,1,7,5.30,5
2019,150,60,58,30,1,13,5.64,6
2020,150,60,58,30,0,5,5.40,5


In [13]:
perf_vietnam = performance_table("vietnam", "headline")
emit(perf_vietnam,
     name="table_7_3b_vietnam_performance_headline",
     number="Table 7.3b",
     title="Vietnam: performance over the headline window",
     placeholder="P22",
     section="§7.3",
     index_header="Portfolio",
     notes=mc_draws_note("vietnam") + PERF_NOTES)

**Table 7.3b. Vietnam: performance over the headline window**  
<sub>§7.3 &middot; fills P22 &middot; `resources/tables/table_7_3b_vietnam_performance_headline.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",11.44%,0.093,0.438,0.085,-70.44%,0.100,26.11%,69.77%,30.0
"Matched random, equal-weight",8.67% ± 2.04%,—,0.328 ± 0.078,—,-72.92% ± 1.90%,—,—,—,—
"F-Score, GMV (RMT-denoised)",18.31%*,0.047,0.763,0.050,-54.77%*,0.013,24.00%,89.62%,7.0
"Matched random, GMV",11.58% ± 4.01%,—,0.485 ± 0.170,—,-65.13% ± 4.83%,—,—,—,—
"F-Score, sector-capped GMV",11.29%,0.450,0.468,0.450,-67.04%,0.320,24.10%,86.91%,8.9
"Matched random, sector-capped GMV",10.78% ± 3.58%,—,0.444 ± 0.149,—,-68.38% ± 2.76%,—,—,—,—
"VN30 (VN30 index, VND)",8.92%,—,0.474,—,-48.14%,—,18.81%,—,—
"VNINDEX (all-share index, VND)",9.68%,—,0.537,—,-45.26%,—,18.03%,—,—


<sub>*Notes.* Monte Carlo draws — F-Score, equal-weight: $N = 1000$; F-Score, GMV (RMT-denoised): $N = 300$; F-Score, sector-capped GMV: $N = 300$. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",11.44%,0.093,0.438,0.085,-70.44%,0.100,26.11%,69.77%,30.0
"Matched random, equal-weight",8.67% $\pm$ 2.04%,—,0.328 $\pm$ 0.078,—,-72.92% $\pm$ 1.90%,—,—,—,—
"F-Score, GMV (RMT-denoised)",18.31%*,0.047,0.763,0.050,-54.77%*,0.013,24.00%,89.62%,7.0
"Matched random, GMV",11.58% $\pm$ 4.01%,—,0.485 $\pm$ 0.170,—,-65.13% $\pm$ 4.83%,—,—,—,—
"F-Score, sector-capped GMV",11.29%,0.450,0.468,0.450,-67.04%,0.320,24.10%,86.91%,8.9
"Matched random, sector-capped GMV",10.78% $\pm$ 3.58%,—,0.444 $\pm$ 0.149,—,-68.38% $\pm$ 2.76%,—,—,—,—
"VN30 (VN30 index, VND)",8.92%,—,0.474,—,-48.14%,—,18.81%,—,—
"VNINDEX (all-share index, VND)",9.68%,—,0.537,—,-45.26%,—,18.03%,—,—


In [14]:
perf_vietnam_full = performance_table("vietnam", "full")
emit(perf_vietnam_full,
     name="table_7_3c_vietnam_performance_full",
     number="Table 7.3c",
     title="Vietnam: performance over the full available window",
     placeholder="P22",
     section="§7.3",
     index_header="Portfolio",
     notes=("Same construction as the previous table over every formation year the data "
            "support; see Table 7.0 for the two windows. " + PERF_NOTES))

**Table 7.3c. Vietnam: performance over the full available window**  
<sub>§7.3 &middot; fills P22 &middot; `resources/tables/table_7_3c_vietnam_performance_full.tex`</sub>

Portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
"F-Score, equal-weight",11.67%*,0.047,0.446*,0.042,-70.44%,0.100,26.14%,70.36%,30.0
"Matched random, equal-weight",8.39% ± 1.92%,—,0.317 ± 0.073,—,-72.92% ± 1.90%,—,—,—,—
"F-Score, GMV (RMT-denoised)",19.30%,0.063,0.807,0.067,-54.77%*,0.010,23.91%,89.96%,6.9
"Matched random, GMV",13.18% ± 3.85%,—,0.550 ± 0.163,—,-65.13% ± 4.81%,—,—,—,—
"F-Score, sector-capped GMV",13.07%,0.390,0.545,0.380,-67.04%,0.320,23.98%,87.21%,8.8
"Matched random, sector-capped GMV",12.02% ± 3.47%,—,0.496 ± 0.146,—,-68.38% ± 2.76%,—,—,—,—


<sub>*Notes.* Same construction as the previous table over every formation year the data support; see Table 7.0 for the two windows. Rows pair each F-Score portfolio with the matched random portfolio drawn from the same annual high-B/M universe under the same construction, so reading across a pair isolates the selection rule and reading down the pairs isolates portfolio construction. Random rows report the Monte Carlo mean $\pm$ standard deviation and carry no test of their own, being the null. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Annualised volatility, turnover and Effective N have no null attached and are therefore never starred. Returns are gross; turnover is one-way.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Annualised volatility,Turnover (one-way),Effective N
Portfolio,,,,,,,,,
"F-Score, equal-weight",11.67%*,0.047,0.446*,0.042,-70.44%,0.100,26.14%,70.36%,30.0
"Matched random, equal-weight",8.39% $\pm$ 1.92%,—,0.317 $\pm$ 0.073,—,-72.92% $\pm$ 1.90%,—,—,—,—
"F-Score, GMV (RMT-denoised)",19.30%,0.063,0.807,0.067,-54.77%*,0.010,23.91%,89.96%,6.9
"Matched random, GMV",13.18% $\pm$ 3.85%,—,0.550 $\pm$ 0.163,—,-65.13% $\pm$ 4.81%,—,—,—,—
"F-Score, sector-capped GMV",13.07%,0.390,0.545,0.380,-67.04%,0.320,23.98%,87.21%,8.8
"Matched random, sector-capped GMV",12.02% $\pm$ 3.47%,—,0.496 $\pm$ 0.146,—,-68.38% $\pm$ 2.76%,—,—,—,—


---
## 8. Cross-Country Comparison

One table, every market, the same strategy definitions. Read the F-Score rows against their
market proxies for the economic comparison and against the p-value column for the statistical
one. The window caveat from Table 7.0 applies to every row.

In [15]:
# ---------------------------------------------------------------------
# Table 8.1 — the cross-country table (P24)
# EW and GMV performance, Monte Carlo p-values, market proxies, turnover,
# maximum weight, non-zero names, Effective N.
# Same layout rule as the country tables: every tested statistic is followed
# by its own p-value against the matched random null.
# ---------------------------------------------------------------------
XC_COLS = ([c for _, label, _ in TESTED for c in (label, "p-value")]
           + ["Turnover (one-way)", "Effective N", "Maximum weight",
              "Non-zero names"])

labels, rows = [], []
for m in MARKETS:
    summ = read_indexed(R.RESULTS / f"{m}_summary.csv")
    plc = read_placement(R.RESULTS / f"{m}_mc_placement.csv")
    if summ is None:
        continue
    for strat in ["fscore_EW", "fscore_GMV"]:
        if strat not in summ.index:
            continue
        s = summ.loc[strat]
        row = []
        for key, _, fmt in TESTED:
            p = _placement(plc, strat, key, "p_value")
            row += [marked(fmt(s[key]), p), num(p, 3)]
        row += [pct(s.turnover), num(s.effective_n, 1),
                DASH,      # maximum weight: not exported by the pipeline
                DASH]      # non-zero names: not exported by the pipeline
        labels.append(f"{MARKET_LABEL[m]} — {STRATEGY_LABEL[strat]}")
        rows.append(row)
    for ix in summ.index:                       # market proxies
        if ix in STRATEGY_LABEL:
            continue
        s = summ.loc[ix]
        row = []
        for key, _, fmt in TESTED:
            row += [fmt(s[key]), DASH]
        row += [DASH] * 4
        labels.append(f"{MARKET_LABEL[m]} — {ix}")
        rows.append(row)

cross_country = pd.DataFrame(rows, index=labels, columns=XC_COLS)
cross_country.index.name = "Market and portfolio"

emit(cross_country,
     name="table_8_1_cross_country",
     number="Table 8.1",
     title="Cross-country comparison: F-Score selection and GMV construction, by market",
     placeholder="P24",
     section="§8",
     index_header="Market and portfolio",
     notes=(P_NOTE + STAR_NOTE +
            "Each market is measured over its own headline window, not a common one — the "
            "United States and Vietnam over 2012--2024, Japan over 2023--2024 (Table 7.0). The "
            "p-values are therefore not strictly comparable across markets, and Japan's "
            "rest on two formation years, one of which held nine names. Maximum weight and "
            "non-zero names are shown as — because the pipeline does not export them. Market "
            "proxies carry no Monte Carlo null and are never starred."))


**Table 8.1. Cross-country comparison: F-Score selection and GMV construction, by market**  
<sub>§8 &middot; fills P24 &middot; `resources/tables/table_8_1_cross_country.tex`</sub>

Market and portfolio,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Turnover (one-way),Effective N,Maximum weight,Non-zero names
"United States — F-Score, equal-weight",15.92%,0.565,0.811,0.527,-38.64%,0.102,60.94%,30.0,—,—
"United States — F-Score, GMV (RMT-denoised)",11.41%,0.087,0.736*,0.030,-29.11%*,0.040,60.96%,5.6,—,—
United States — SPY (S&P 500),14.38%,—,0.847,—,-33.72%,—,—,—,—,—
United States — VTV (US value ETF),12.10%,—,0.756,—,-36.78%,—,—,—,—,—
"Japan — F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,96.67%,19.5,—,—
"Japan — F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,100.00%,6.5,—,—
"Japan — 1306.T (TOPIX ETF, JPY)",13.96%,—,0.651,—,-22.82%,—,—,—,—,—
"Japan — EWJV (MSCI Japan Value ETF, USD)",16.79%,—,0.912,—,-14.61%,—,—,—,—,—
"Vietnam — F-Score, equal-weight",11.44%,0.093,0.438,0.085,-70.44%,0.100,69.77%,30.0,—,—
"Vietnam — F-Score, GMV (RMT-denoised)",18.31%*,0.047,0.763,0.050,-54.77%*,0.013,89.62%,7.0,—,—


<sub>*Notes.* Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. Each market is measured over its own headline window, not a common one — the United States and Vietnam over 2012--2024, Japan over 2023--2024 (Table 7.0). The p-values are therefore not strictly comparable across markets, and Japan's rest on two formation years, one of which held nine names. Maximum weight and non-zero names are shown as — because the pipeline does not export them. Market proxies carry no Monte Carlo null and are never starred.</sub>

,Annualised return,p-value,Sharpe ratio,p-value,Maximum drawdown,p-value,Turnover (one-way),Effective N,Maximum weight,Non-zero names
Market and portfolio,,,,,,,,,,
"United States — F-Score, equal-weight",15.92%,0.565,0.811,0.527,-38.64%,0.102,60.94%,30.0,—,—
"United States — F-Score, GMV (RMT-denoised)",11.41%,0.087,0.736*,0.030,-29.11%*,0.040,60.96%,5.6,—,—
United States — SPY (S&P 500),14.38%,—,0.847,—,-33.72%,—,—,—,—,—
United States — VTV (US value ETF),12.10%,—,0.756,—,-36.78%,—,—,—,—,—
"Japan — F-Score, equal-weight",17.98%,0.214,0.766,0.251,-23.98%,0.751,96.67%,19.5,—,—
"Japan — F-Score, GMV (RMT-denoised)",20.88%,0.280,1.030,0.330,-15.98%,0.530,100.00%,6.5,—,—
"Japan — 1306.T (TOPIX ETF, JPY)",13.96%,—,0.651,—,-22.82%,—,—,—,—,—
"Japan — EWJV (MSCI Japan Value ETF, USD)",16.79%,—,0.912,—,-14.61%,—,—,—,—,—
"Vietnam — F-Score, equal-weight",11.44%,0.093,0.438,0.085,-70.44%,0.100,69.77%,30.0,—,—


---
## 9. Robustness and Alternative Specifications

In [16]:
# ---------------------------------------------------------------------
# Table 9.1 — the K x N grid (P26)
# Sharpe is invariant to N (N changes only the resolution of the null), so the
# three N columns carry the p-values and show Monte Carlo convergence.
# ---------------------------------------------------------------------
parts = [read(R.RESULTS / "grid_summary_2012_2024.csv")]
for m in MARKETS:
    parts.append(read(R.GRID / f"{m}_grid_summary.csv"))
grid = pd.concat([p for p in parts if p is not None], ignore_index=True)
grid = grid.drop_duplicates(subset=["mkt", "k", "N"], keep="first")


rows = {}
for (mkt, k), g in grid.groupby(["mkt", "k"], sort=False):
    g = g.set_index("N")
    first = g.iloc[0]
    rec = {
        "F-Score EW Sharpe": marked(num(first.EW), g.loc[1000, "p"] if 1000 in g.index else None),
        "Whole-universe EW Sharpe": num(first.uniEW),
        "Highest-B/M EW Sharpe": num(first.value),
        "Turnover (one-way)": pct(first.to_EW),
        "Effective N (GMV)": num(first.effN_GMV, 1),
    }
    for n in (1000, 2000, 5000):
        rec[f"p-value ($N$ = {n})"] = pval(g.loc[n, "p"]) if n in g.index else DASH
    rec["GMV $-$ EW spread"] = marked(num(first.D), first.D_p)
    rec["p-value (spread)"] = pval(first.D_p)
    rows[f"{MARKET_LABEL.get(mkt, mkt)} — $K$ = {int(k)}"] = rec

grid_table = pd.DataFrame(rows).T
grid_table.index.name = "Market and basket size"

emit(grid_table,
     name="table_9_1_grid_k_by_n",
     number="Table 9.1",
     title="Basket size and Monte Carlo convergence: the $K \\times N$ grid",
     placeholder="P26",
     section="§9.1",
     index_header="Market and basket size",
     notes=(P_NOTE + STAR_NOTE +
            "The full $3 \\times 3$ grid was run in all three markets over the common "
            "2012--2024 formation window. The Sharpe ratio does not depend on $N$, which "
            "changes only the resolution of the null, so the three p-value columns show Monte "
            "Carlo convergence rather than three different experiments; the Sharpe ratio is "
            "starred on its $N = 1000$ p-value. These runs use the whole scoreable universe, "
            "not the high-B/M "
            "subset, and so answer the broad-universe question of §9.2 rather than the "
            "headline design. The GMV $-$ EW spread is the incremental value of construction "
            "over equal weighting on the same basket, tested against its own null."))


**Table 9.1. Basket size and Monte Carlo convergence: the $K \times N$ grid**  
<sub>§9.1 &middot; fills P26 &middot; `resources/tables/table_9_1_grid_k_by_n.tex`</sub>

Market and basket size,F-Score EW Sharpe,Whole-universe EW Sharpe,Highest-B/M EW Sharpe,Turnover (one-way),Effective N (GMV),p-value ($N$ = 1000),p-value ($N$ = 2000),p-value ($N$ = 5000),GMV $-$ EW spread,p-value (spread)
United States — $K$ = 20,0.735,0.857,0.779,72.39%,5.7,0.838,0.839,0.838,-0.216,0.400
United States — $K$ = 25,0.763,0.857,0.821,68.65%,6.6,0.782,0.792,0.810,-0.197,0.283
United States — $K$ = 30,0.810,0.857,0.827,61.76%,7.5,0.644,0.641,0.643,-0.178,0.197
Japan — $K$ = 20,0.742,0.720,0.690,75.20%,4.7,0.270,0.255,0.257,-0.203,0.990
Japan — $K$ = 25,0.706,0.720,0.721,71.18%,5.3,0.462,0.479,0.472,-0.086,0.860
Japan — $K$ = 30,0.701,0.720,0.718,66.91%,5.7,0.525,0.542,0.537,0.015,0.703
Vietnam — $K$ = 20,1.286*,1.285,0.594,89.58%,10.1,0.010*,0.014*,0.016*,0.227,0.500
Vietnam — $K$ = 25,1.318*,1.285,0.662,89.51%,11.5,0.010*,0.013*,0.015*,0.451,0.140
Vietnam — $K$ = 30,1.184,1.285,0.657,89.43%,13.7,0.108,0.110,0.105,0.418,0.253


<sub>*Notes.* Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. The full $3 \times 3$ grid was run in all three markets over the common 2012--2024 formation window. The Sharpe ratio does not depend on $N$, which changes only the resolution of the null, so the three p-value columns show Monte Carlo convergence rather than three different experiments; the Sharpe ratio is starred on its $N = 1000$ p-value. These runs use the whole scoreable universe, not the high-B/M subset, and so answer the broad-universe question of §9.2 rather than the headline design. The GMV $-$ EW spread is the incremental value of construction over equal weighting on the same basket, tested against its own null.</sub>

,F-Score EW Sharpe,Whole-universe EW Sharpe,Highest-B/M EW Sharpe,Turnover (one-way),Effective N (GMV),p-value ($N$ = 1000),p-value ($N$ = 2000),p-value ($N$ = 5000),GMV $-$ EW spread,p-value (spread)
Market and basket size,,,,,,,,,,
United States — $K$ = 20,0.735,0.857,0.779,72.39%,5.7,0.838,0.839,0.838,-0.216,0.400
United States — $K$ = 25,0.763,0.857,0.821,68.65%,6.6,0.782,0.792,0.810,-0.197,0.283
United States — $K$ = 30,0.810,0.857,0.827,61.76%,7.5,0.644,0.641,0.643,-0.178,0.197
Japan — $K$ = 20,0.742,0.720,0.690,75.20%,4.7,0.270,0.255,0.257,-0.203,0.990
Japan — $K$ = 25,0.706,0.720,0.721,71.18%,5.3,0.462,0.479,0.472,-0.086,0.860
Japan — $K$ = 30,0.701,0.720,0.718,66.91%,5.7,0.525,0.542,0.537,0.015,0.703
Vietnam — $K$ = 20,1.286*,1.285,0.594,89.58%,10.1,0.010*,0.014*,0.016*,0.227,0.500
Vietnam — $K$ = 25,1.318*,1.285,0.662,89.51%,11.5,0.010*,0.013*,0.015*,0.451,0.140
Vietnam — $K$ = 30,1.184,1.285,0.657,89.43%,13.7,0.108,0.110,0.105,0.418,0.253


In [17]:
# ---------------------------------------------------------------------
# Table 9.2 — broad-universe F-Score (P27)
# The grid family: F-Score applied to the whole scoreable universe, with no
# high-B/M filter first. Two nulls are reported, each with its own p-value.
# ---------------------------------------------------------------------
BROAD_ORDER = ["fscore_EW", "fscore_GMV", "value_EW", "universe_EW", "universe_GMV"]
CELL = "k25_mc1000"
NULLS = [("random (full universe)", "vs random (full universe)"),
         ("random (non-F-Score names)", "vs random (non-F-Score names)")]

BROAD_COLS = (["Annualised return", "Annualised volatility", "Sharpe ratio",
               "Maximum drawdown", "Turnover (one-way)", "Effective N"]
              + [f"Sharpe: p {label}" for _, label in NULLS])

labels, rows = [], []
for m in MARKETS:
    summ = read_indexed(R.GRID / f"{m}_{CELL}_summary.csv")
    plc = read(R.GRID / f"{m}_{CELL}_placement.csv")
    if summ is None:
        continue
    if plc is not None:
        # the file carries three unnamed index columns; the third repeats the
        # `basis` column already present, so only the first two are renamed
        plc = plc.rename(columns={plc.columns[0]: "null", plc.columns[1]: "metric"})
        plc = plc.loc[:, ~plc.columns.duplicated()]
        plc = plc[plc["basis"] == "gross"].set_index(["null", "metric"])

    for strat in BROAD_ORDER:
        if strat not in summ.index:
            continue
        s = summ.loc[strat]
        row = [pct(s.ann_return), pct(s.ann_vol), num(s.sharpe),
               pct(s.max_drawdown), pct(s.turnover), num(s.effective_n, 1)]
        for null_name, _ in NULLS:
            # only the F-Score equal-weight basket is placed against these nulls
            if strat != "fscore_EW" or plc is None or (null_name, "sharpe") not in plc.index:
                row.append(DASH)
                continue
            # pval() bolds and stars on its own value
            row.append(pval(plc.loc[(null_name, "sharpe"), "p_value"]))
        labels.append(f"{MARKET_LABEL[m]} — {STRATEGY_LABEL[strat]}")
        rows.append(row)

broad_table = pd.DataFrame(rows, index=labels, columns=BROAD_COLS)
broad_table.index.name = "Market and portfolio"

emit(broad_table,
     name="table_9_2_broad_universe",
     number="Table 9.2",
     title="Broad-universe F-Score: no high-B/M filter applied first",
     placeholder="P27",
     section="§9.2",
     index_header="Market and portfolio",
     notes=(f"Grid cell $K = 25$, $N = 1000$, formation years 2012--2024 in all three markets. "
            + P_NOTE + STAR_NOTE +
            "The last two columns give the Sharpe ratio's p-value against two different nulls — "
            "random baskets from the whole universe, and random baskets drawn only from names "
            "the F-Score did not pick; both are gross, and only the equal-weight F-Score basket "
            "is placed against them. This specification answers a "
            "different question from the headline design: whether the F-Score works as a "
            "general selection signal outside the value setting it was built for. It must not "
            "be mixed with the high-B/M null of §7. Note also that the whole-universe rows are "
            "dominated by names with very little traded volume, so this table measures "
            "information content rather than an investable result."))


**Table 9.2. Broad-universe F-Score: no high-B/M filter applied first**  
<sub>§9.2 &middot; fills P27 &middot; `resources/tables/table_9_2_broad_universe.tex`</sub>

Market and portfolio,Annualised return,Annualised volatility,Sharpe ratio,Maximum drawdown,Turnover (one-way),Effective N,Sharpe: p vs random (full universe),Sharpe: p vs random (non-F-Score names)
"United States — F-Score, equal-weight",11.94%,15.65%,0.763,-34.32%,68.65%,25.0,0.782,0.919
"United States — F-Score, GMV (RMT-denoised)",8.56%,15.13%,0.566,-35.76%,83.86%,6.6,—,—
"United States — Highest B/M, equal-weight",13.86%,16.88%,0.821,-36.81%,25.00%,25.0,—,—
"United States — Whole universe, equal-weight",13.31%,15.53%,0.857,-32.79%,10.94%,72.4,—,—
"United States — Whole universe, GMV",7.51%,14.33%,0.524,-36.46%,35.61%,8.8,—,—
"Japan — F-Score, equal-weight",14.09%,19.97%,0.706,-36.05%,71.18%,25.0,0.462,0.447
"Japan — F-Score, GMV (RMT-denoised)",10.99%,17.75%,0.619,-44.00%,78.87%,5.3,—,—
"Japan — Highest B/M, equal-weight",14.28%,19.81%,0.721,-31.76%,26.74%,65.9,—,—
"Japan — Whole universe, equal-weight",13.89%,19.29%,0.720,-31.76%,11.18%,80.5,—,—
"Japan — Whole universe, GMV",13.16%,16.13%,0.816,-40.73%,34.08%,6.2,—,—


<sub>*Notes.* Grid cell $K = 25$, $N = 1000$, formation years 2012--2024 in all three markets. Each tested statistic is followed by its one-sided empirical p-value against the matched random null: the share of Monte Carlo draws that equalled or beat the F-Score portfolio on that statistic. Maximum drawdown is a negative return, so its test treats a shallower drawdown as better. An asterisk marks significance at the 5% level, the single level fixed in advance in §6; each statistic is starred on its own empirical p-value, not on the Sharpe ratio's. The last two columns give the Sharpe ratio's p-value against two different nulls — random baskets from the whole universe, and random baskets drawn only from names the F-Score did not pick; both are gross, and only the equal-weight F-Score basket is placed against them. This specification answers a different question from the headline design: whether the F-Score works as a general selection signal outside the value setting it was built for. It must not be mixed with the high-B/M null of §7. Note also that the whole-universe rows are dominated by names with very little traded volume, so this table measures information content rather than an investable result.</sub>

,Annualised return,Annualised volatility,Sharpe ratio,Maximum drawdown,Turnover (one-way),Effective N,Sharpe: p vs random (full universe),Sharpe: p vs random (non-F-Score names)
Market and portfolio,,,,,,,,
"United States — F-Score, equal-weight",11.94%,15.65%,0.763,-34.32%,68.65%,25.0,0.782,0.919
"United States — F-Score, GMV (RMT-denoised)",8.56%,15.13%,0.566,-35.76%,83.86%,6.6,—,—
"United States — Highest B/M, equal-weight",13.86%,16.88%,0.821,-36.81%,25.00%,25.0,—,—
"United States — Whole universe, equal-weight",13.31%,15.53%,0.857,-32.79%,10.94%,72.4,—,—
"United States — Whole universe, GMV",7.51%,14.33%,0.524,-36.46%,35.61%,8.8,—,—
"Japan — F-Score, equal-weight",14.09%,19.97%,0.706,-36.05%,71.18%,25.0,0.462,0.447
"Japan — F-Score, GMV (RMT-denoised)",10.99%,17.75%,0.619,-44.00%,78.87%,5.3,—,—
"Japan — Highest B/M, equal-weight",14.28%,19.81%,0.721,-31.76%,26.74%,65.9,—,—
"Japan — Whole universe, equal-weight",13.89%,19.29%,0.720,-31.76%,11.18%,80.5,—,—


In [18]:
# ---------------------------------------------------------------------
# Table 9.3 — status of the remaining robustness specifications
# Fills P28 (delisting), P29 (sector-constrained GMV), P30 (economic regimes)
# by stating plainly what was and was not run.
# ---------------------------------------------------------------------
status = pd.DataFrame(
    [
        ("Basket size and Monte Carlo grid",
         "Run", "$K \\in \\{20,25,30\\} \\times N \\in \\{1000,2000,5000\\}$, all three markets",
         "Table 9.1"),
        ("Broad-universe F-Score",
         "Run", "Whole scoreable universe, 2012--2024, all three markets",
         "Table 9.2"),
        ("Delisting at $-100\\%$",
         sig("Not run", True),
         "The main convention (exit at the last tradable price with volume) is the only one "
         "implemented",
         "P28 must say so explicitly"),
        ("Sector-constrained GMV",
         "Run, not yet placed",
         "A 20% sector cap is produced in all three markets as fscore_GMVsec",
         "Tables 7.1b--7.3b; P10/P29 need the headline decision"),
        ("Economic regimes",
         sig("Not rerun", True),
         "The earlier COVID and recovery subperiod work used a superseded United States "
         "specification",
         "P30: rerun under the unified design or drop the subsection"),
        ("Tie-break sensitivity",
         "Run (Vietnam only)",
         "Eight seeds, reported in vietnam_tiebreak_sensitivity.csv",
         "Extend to the other two markets or state the limitation"),
        ("Full-window robustness",
         "Run", "Each market over every formation year its data support",
         "Tables 7.1c--7.3c"),
    ],
    columns=["Specification", "Status", "What exists", "Where it goes"],
).set_index("Specification")

emit(status,
     name="table_9_3_robustness_status",
     number="Table 9.3",
     title="Status of the robustness specifications named in the draft",
     placeholder="P26, P27, P28, P29, P30",
     section="§9",
     index_header="Specification",
     notes=("Starred rows are specifications the draft reserves space for but which have not "
            "been run under the unified pipeline. The paper should state that plainly rather "
            "than leave the subsection empty."))

**Table 9.3. Status of the robustness specifications named in the draft**  
<sub>§9 &middot; fills P26, P27, P28, P29, P30 &middot; `resources/tables/table_9_3_robustness_status.tex`</sub>

Specification,Status,What exists,Where it goes
Basket size and Monte Carlo grid,Run,"$K \in \{20,25,30\} \times N \in \{1000,2000,5000\}$, all three markets",Table 9.1
Broad-universe F-Score,Run,"Whole scoreable universe, 2012--2024, all three markets",Table 9.2
Delisting at $-100\%$,Not run*,The main convention (exit at the last tradable price with volume) is the only one implemented,P28 must say so explicitly
Sector-constrained GMV,"Run, not yet placed",A 20% sector cap is produced in all three markets as fscore_GMVsec,Tables 7.1b--7.3b; P10/P29 need the headline decision
Economic regimes,Not rerun*,The earlier COVID and recovery subperiod work used a superseded United States specification,P30: rerun under the unified design or drop the subsection
Tie-break sensitivity,Run (Vietnam only),"Eight seeds, reported in vietnam_tiebreak_sensitivity.csv",Extend to the other two markets or state the limitation
Full-window robustness,Run,Each market over every formation year its data support,Tables 7.1c--7.3c


<sub>*Notes.* Starred rows are specifications the draft reserves space for but which have not been run under the unified pipeline. The paper should state that plainly rather than leave the subsection empty.</sub>

,Status,What exists,Where it goes
Specification,,,
Basket size and Monte Carlo grid,Run,"$K \in \{20,25,30\} \times N \in \{1000,2000,5...",Table 9.1
Broad-universe F-Score,Run,"Whole scoreable universe, 2012--2024, all thre...",Table 9.2
Delisting at $-100\%$,Not run*,The main convention (exit at the last tradable...,P28 must say so explicitly
Sector-constrained GMV,"Run, not yet placed",A 20% sector cap is produced in all three mark...,Tables 7.1b--7.3b; P10/P29 need the headline d...
Economic regimes,Not rerun*,The earlier COVID and recovery subperiod work ...,P30: rerun under the unified design or drop th...
Tie-break sensitivity,Run (Vietnam only),"Eight seeds, reported in vietnam_tiebreak_sens...",Extend to the other two markets or state the l...
Full-window robustness,Run,Each market over every formation year its data...,Tables 7.1c--7.3c


---
## 11. Limitations and Data Integrity

Three tables. The first states what each market's data actually is; the second counts what was
dropped and why; the third explains, year by year, which formation years the data could support.
Together they fill P32 and supply the evidence for the limitations section.

In [19]:
# ---------------------------------------------------------------------
# Table 11.1 — data source and coverage by country (P32)
# Facts taken from data/README.md, results/panel/PROVENANCE.md and the
# loaders; each is checkable against the repository.
# ---------------------------------------------------------------------
quality = pd.DataFrame(
    [
        ("Fundamentals source",
         "SEC EDGAR (10-K first prints) plus Yahoo Finance",
         "Licensed-terminal statements, reduced to nine flags",
         "Sibling preprocessing repository: FireAnt, CafeF and TCBS reconciled"),
        ("Report date basis",
         sig("Actual 10-K filing date", True),
         "Fiscal year end, 31 March",
         "Fiscal year end, 31 December"),
        ("Reporting lag applied",
         "5 months", "5 months", "5 months (documented as 6 in the data notes)"),
        ("Price source",
         "Yahoo Finance, adjusted closes", "Yahoo Finance, adjusted closes",
         "Sibling repository, dividend-adjusted closes"),
        ("Market proxy",
         "SPY, VTV (total-return ETFs)",
         "1306.T, EWJV (total-return ETFs)",
         sig("VN30, VNINDEX (capital indices, no dividends)", True)),
        ("Upstream liquidity screen",
         "None", "None",
         sig("June-turnover tradability gate, applied before this repository sees the data", True)),
        ("Delisted coverage",
         "Partial", "Partial",
         "Partial: 125 of 1,371 tickers stop printing before 2026"),
        ("Short selling",
         "Available", "Available", "Not available; Vietnam runs long-only"),
    ],
    columns=["Item", "United States", "Japan", "Vietnam"],
).set_index("Item")

emit(quality,
     name="table_11_1_data_quality",
     number="Table 11.1",
     title="Data source and coverage, by country",
     placeholder="P32",
     section="§11",
     index_header="Item",
     notes=("Starred cells are the differences that survive the unified pipeline because they "
            "live upstream of it. Three matter for interpretation: only the United States has "
            "genuine point-in-time report dates, the other two using a fiscal period end plus a "
            "fixed lag; Vietnam's benchmarks exclude cash dividends, which flatters every "
            "portfolio-against-index comparison by roughly 1.5--2% a year; and Vietnam's panel "
            "arrives already liquidity-screened, a filter the other two markets never receive."))

**Table 11.1. Data source and coverage, by country**  
<sub>§11 &middot; fills P32 &middot; `resources/tables/table_11_1_data_quality.tex`</sub>

Item,United States,Japan,Vietnam
Fundamentals source,SEC EDGAR (10-K first prints) plus Yahoo Finance,"Licensed-terminal statements, reduced to nine flags","Sibling preprocessing repository: FireAnt, CafeF and TCBS reconciled"
Report date basis,Actual 10-K filing date*,"Fiscal year end, 31 March","Fiscal year end, 31 December"
Reporting lag applied,5 months,5 months,5 months (documented as 6 in the data notes)
Price source,"Yahoo Finance, adjusted closes","Yahoo Finance, adjusted closes","Sibling repository, dividend-adjusted closes"
Market proxy,"SPY, VTV (total-return ETFs)","1306.T, EWJV (total-return ETFs)","VN30, VNINDEX (capital indices, no dividends)*"
Upstream liquidity screen,None,None,"June-turnover tradability gate, applied before this repository sees the data*"
Delisted coverage,Partial,Partial,"Partial: 125 of 1,371 tickers stop printing before 2026"
Short selling,Available,Available,Not available; Vietnam runs long-only


<sub>*Notes.* Starred cells are the differences that survive the unified pipeline because they live upstream of it. Three matter for interpretation: only the United States has genuine point-in-time report dates, the other two using a fiscal period end plus a fixed lag; Vietnam's benchmarks exclude cash dividends, which flatters every portfolio-against-index comparison by roughly 1.5--2% a year; and Vietnam's panel arrives already liquidity-screened, a filter the other two markets never receive.</sub>

,United States,Japan,Vietnam
Item,,,
Fundamentals source,SEC EDGAR (10-K first prints) plus Yahoo Finance,"Licensed-terminal statements, reduced to nine ...","Sibling preprocessing repository: FireAnt, Caf..."
Report date basis,Actual 10-K filing date*,"Fiscal year end, 31 March","Fiscal year end, 31 December"
Reporting lag applied,5 months,5 months,5 months (documented as 6 in the data notes)
Price source,"Yahoo Finance, adjusted closes","Yahoo Finance, adjusted closes","Sibling repository, dividend-adjusted closes"
Market proxy,"SPY, VTV (total-return ETFs)","1306.T, EWJV (total-return ETFs)","VN30, VNINDEX (capital indices, no dividends)*"
Upstream liquidity screen,None,None,"June-turnover tradability gate, applied befor..."
Delisted coverage,Partial,Partial,"Partial: 125 of 1,371 tickers stop printing be..."
Short selling,Available,Available,Not available; Vietnam runs long-only


In [20]:
# ---------------------------------------------------------------------
# Table 11.2 — what the source held against what was scored (P32)
# Drop reasons differ by market because the pipelines upstream differ; the
# union is shown with — where a market has no such stage.
# ---------------------------------------------------------------------
REASON_LABEL = {
    "rows_in_source": "Firm-years in source",
    "dropped_failed_accounting_checks": "Dropped: failed accounting checks",
    "dropped_no_prior_year": "Dropped: no prior year to compare",
    "dropped_incomplete_signals": "Dropped: incomplete F-Score inputs",
    "dropped_unresolved_identifier": "Dropped: unresolved identifier",
    "dropped_no_book_to_market": "Dropped: no book-to-market",
    "dropped_no_june_turnover": "Dropped: no June turnover",
    "dropped_no_formation_price": "Dropped: no formation price",
    "dropped_below_liquidity_gate": "Dropped: below the liquidity gate",
    "scored": "Scoreable firm-years",
    "pct_scored": "Share of source scored",
}

excl = {}
for m in MARKETS:
    df = read(R.GRID / f"{m}_{CELL}_exclusions.csv")
    if df is None:
        continue
    row = df.set_index(df.columns[0]).iloc[0]
    excl[MARKET_LABEL[m]] = row

exclusions = pd.DataFrame(index=list(REASON_LABEL), columns=list(excl))
for mk, row in excl.items():
    for key in REASON_LABEL:
        if key not in row.index or pd.isna(row.get(key)):
            exclusions.loc[key, mk] = DASH
        elif key == "pct_scored":
            exclusions.loc[key, mk] = f"{float(row[key]):.1f}%"
        else:
            exclusions.loc[key, mk] = f"{int(row[key]):,}"
exclusions.index = [REASON_LABEL[k] for k in exclusions.index]
exclusions.index.name = "Stage"

emit(exclusions,
     name="table_11_2_exclusions",
     number="Table 11.2",
     title="Source firm-years against scoreable firm-years, by country",
     placeholder="P32",
     section="§11",
     index_header="Stage",
     notes=("A — means the market has no such stage, not that nothing was dropped. Vietnam "
            "scores 40.4% of its source rows against 83.0% and 88.7%, most of the difference "
            "being the upstream liquidity gate and the accounting checks that the other two "
            "panels never apply. The share scored is therefore not a data-quality ranking."))

**Table 11.2. Source firm-years against scoreable firm-years, by country**  
<sub>§11 &middot; fills P32 &middot; `resources/tables/table_11_2_exclusions.tex`</sub>

Stage,United States,Japan,Vietnam
Firm-years in source,"2,096","2,116","23,493"
Dropped: failed accounting checks,—,—,"2,260"
Dropped: no prior year to compare,208,235,"2,388"
Dropped: incomplete F-Score inputs,0,0,"2,281"
Dropped: unresolved identifier,149,5,0
Dropped: no book-to-market,—,—,"1,691"
Dropped: no June turnover,—,—,50
Dropped: no formation price,—,—,45
Dropped: below the liquidity gate,—,—,"5,296"
Scoreable firm-years,"1,739","1,876","9,482"


<sub>*Notes.* A — means the market has no such stage, not that nothing was dropped. Vietnam scores 40.4% of its source rows against 83.0% and 88.7%, most of the difference being the upstream liquidity gate and the accounting checks that the other two panels never apply. The share scored is therefore not a data-quality ranking.</sub>

,United States,Japan,Vietnam
Stage,,,
Firm-years in source,"2,096","2,116","23,493"
Dropped: failed accounting checks,—,—,"2,260"
Dropped: no prior year to compare,208,235,"2,388"
Dropped: incomplete F-Score inputs,0,0,"2,281"
Dropped: unresolved identifier,149,5,0
Dropped: no book-to-market,—,—,"1,691"
Dropped: no June turnover,—,—,50
Dropped: no formation price,—,—,45
Dropped: below the liquidity gate,—,—,"5,296"


In [21]:
# ---------------------------------------------------------------------
# Table 11.3 — why each market's window is what it is (P32, and the
# evidence behind Table 7.0)
# ---------------------------------------------------------------------
feas = {}
for m in MARKETS:
    df = read(R.RESULTS / f"{m}_fullperiod_feasibility.csv")
    if df is None:
        continue
    usable = df[df.usable == True]                                   # noqa: E712
    reasons = df[df.usable != True].reason.fillna("").value_counts()  # noqa: E712
    feas[MARKET_LABEL[m]] = {
        "Formation years examined": integer(len(df)),
        "Usable": sig(integer(len(usable)), len(usable) < 5),
        "First usable": integer(usable.year.min()) if len(usable) else DASH,
        "Last usable": integer(usable.year.max()) if len(usable) else DASH,
        "Blocked: no point-in-time statements":
            integer(reasons.get("no point-in-time statements", 0)),
        "Blocked: holding year past the evaluation end":
            integer(sum(v for k, v in reasons.items() if "evaluation end" in k)),
        "Blocked: other": integer(sum(
            v for k, v in reasons.items()
            if k != "no point-in-time statements" and "evaluation end" not in k)),
    }

feasibility = pd.DataFrame(feas).T
feasibility.index.name = "Market"

emit(feasibility,
     name="table_11_3_feasibility",
     number="Table 11.3",
     title="Formation-year feasibility, by country",
     placeholder="P32, P03",
     section="§11, §3.6",
     index_header="Market",
     notes=("This is the evidence behind Table 7.0. Japan's two usable formation years are not "
            "a modelling choice: the fundamentals cache carries no point-in-time statements "
            "before fiscal 2022, so every earlier formation is blocked at the data layer. Years "
            "blocked because the holding year would end past 30 June 2025 are a property of the "
            "evaluation window, not of the data."))

**Table 11.3. Formation-year feasibility, by country**  
<sub>§11, §3.6 &middot; fills P32, P03 &middot; `resources/tables/table_11_3_feasibility.tex`</sub>

Market,Formation years examined,Usable,First usable,Last usable,Blocked: no point-in-time statements,Blocked: holding year past the evaluation end,Blocked: other
United States,22,15,2010,2024,5,2,0
Japan,22,2*,2023,2024,18,2,0
Vietnam,22,14,2011,2024,6,2,0


<sub>*Notes.* This is the evidence behind Table 7.0. Japan's two usable formation years are not a modelling choice: the fundamentals cache carries no point-in-time statements before fiscal 2022, so every earlier formation is blocked at the data layer. Years blocked because the holding year would end past 30 June 2025 are a property of the evaluation window, not of the data.</sub>

,Formation years examined,Usable,First usable,Last usable,Blocked: no point-in-time statements,Blocked: holding year past the evaluation end,Blocked: other
Market,,,,,,,
United States,22,15,2010,2024,5,2,0
Japan,22,2*,2023,2024,18,2,0
Vietnam,22,14,2011,2024,6,2,0


---
## Figures

Copied into `resources/figures/` under names that match their tables. The review companion
audits which of the figures §7 asks for the repository actually produces.

In [22]:
# ---------------------------------------------------------------------
# Figures for P17 (US), P20 (Japan), P23 (Vietnam), plus §9.1 sensitivity.
# The plan lives in report_lib so the review companion can audit the same list.
# ---------------------------------------------------------------------
copied = 0
for market, src_name, dest_name, what in R.FIG_PLAN:
    section = "§9.1" if "grid" in src_name else f"§7.{MARKETS.index(market) + 1}"
    if copy_figure(src_name, dest_name, number=dest_name,
                   title=f"{MARKET_LABEL[market]}: {what}",
                   placeholder=R.FIG_PLACEHOLDER[market], section=section):
        copied += 1

print(f"{copied} of {len(R.FIG_PLAN)} figures copied into resources/figures/")

15 of 15 figures copied into resources/figures/


---
## LaTeX build file

In [23]:
# ---------------------------------------------------------------------
# One \input file so the LaTeX document can pull in every table with a
# single line. Only paper tables appear here — the review companion writes
# to resources/notes/ and is deliberately excluded.
# ---------------------------------------------------------------------
manifest = pd.DataFrame(R.MANIFEST)
tables_only = manifest[manifest.File.str.startswith("tables/")]

inputs = ["% auto-generated by consolidated_report.ipynb",
          "% \\input{resources/_all_tables} to pull in every table at once",
          "% (or \\input them individually, in the order the sections need)",
          "% requires: booktabs, array, graphicx, amsmath"]
for _, r in tables_only.iterrows():
    inputs.append(f"\\input{{resources/{r.File[:-4]}}}   % {r.Table}: {r.Title}")
(R.RES / "_all_tables.tex").write_text("\n".join(inputs) + "\n", encoding="utf-8")

print(f"{len(tables_only)} tables written to resources/tables/")
print(f"{len(manifest) - len(tables_only)} figures written to resources/figures/")
print("resources/_all_tables.tex ready to \\input")

19 tables written to resources/tables/
15 figures written to resources/figures/
resources/_all_tables.tex ready to \input
